# Plot `met_forcing_rxn` simulations

Compare control and enhanced rock weathering simulations run with long-term mean, monthly, daily, and hourly meteorological forcing.

The notebook focuses on drainage and alkalinity export, final geochemical profiles, mean soil-water and gas profiles, and near-surface diffusive gas efflux.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from byte_util.util import all_sites, all_forcing_types, calc_specific_discharge
from byte_util.met_transport import calc_tracer_flux

sim_root = Path('../min3p_runs')

In [ ]:
from min3p.output import read_min3p

def plot_gas_tracer_mass_balance(site, scenario, forcing, skip_records=0):
    """Plot mass balance for the gas tracer.

    skip_records : int
        Number of initial records to skip (to avoid very large values at the start of spin up)"""

    if forcing == 'spinup' or scenario == 'spinup':
        sim_folder = sim_root / site / 'spinup'
        sim_name = 'spinup'
    else:
        sim_folder = sim_root / site / f'{forcing}_{scenario}'
        sim_name = forcing

    # mas : total mol of aqueous species
    mas, _ = read_min3p(sim_folder / f'{sim_name}_o.mas', ftype='transient', fmt='pandas')
    # mgs : total mol of gaseous species
    mgs, _ = read_min3p(sim_folder / f'{sim_name}_o.mgs', ftype='transient', fmt='pandas')
    # mgc : gas mass balance
    mgc, _ = read_min3p(sim_folder / f'{sim_name}_2.mgc', ftype='transient', fmt='pandas')
    mac, _ = read_min3p(sim_folder / f'{sim_name}_12.mac', ftype='transient', fmt='pandas')
    mmc, _ = read_min3p(sim_folder / f'{sim_name}_1.mmc', ftype='transient', fmt='pandas')
    mas.drop(0, inplace=True)
    mgs.drop(0, inplace=True)
    mas.index = np.arange(len(mas))
    mgs.index = np.arange(len(mgs))

    mask = mas['time'].duplicated()
    for df in [mas, mgs, mgc, mac, mmc]:
        # Remove duplicate timesteps (happens when dt is small relative to output precision
        df.drop(df.index[mask], inplace=True)
        df.index = np.arange(len(df))

        # If requested, skip initial records
        if skip_records > 0:
            df.drop(range(0, skip_records), inplace=True)

    time_yr = mas['time'].values/365.25

    fig, ax = plt.subplots(2, 2, figsize=(10, 8), sharex='all')

    # Plot total storage
    ax[0, 0].plot(time_yr, mas['gtr(aq)'], label='Aqueous')
    ax[0, 0].plot(time_yr, mgs['gtr(g)'], label='Gaseous')
    ax[0, 0].plot(time_yr, mas['gtr(aq)'] + mgs['gtr(g)'], label='Total', color='k', ls='--')
    ax[0, 0].set(ylabel='mol', title='Total in domain')
    ax[0, 0].legend()

    # Plot total production
    molar_fraction = 1e-6
    ax[0, 1].plot(time_yr, -mmc['source/sink - aqueous phase [mol/d]']*molar_fraction, label='Mineral dissolution', color='k')
    ax[0, 1].plot(time_yr, mac['source/sink from mineral phase [mol/d]'], label='Aqueous production (mac)', color='0.5', ls='--')
    ax[0, 1].set(ylabel='mol/d', title='Production rate from co2_resp')
    ax[0, 1].annotate('(All three curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')
    ax[0, 1].legend()

    # Calculate change in aqueous storage
    ax[1, 0].plot(time_yr, mac['change in storage [mol/d]'], color='k', label='From mac file')
    ax[1, 0].plot(time_yr[1:], np.diff(mas['gtr(aq)'].values)/np.diff(time_yr*365.25), color='0.5', linestyle='--', label='np.diff(mas)')
    # Inputs = inflow from top + dissolution rate
    aqueous_in = mac['mass influx [mol/d]'] + mac['source/sink from mineral phase [mol/d]']
    # Outputs = outflow from bottom + loss to gas phase
    aqueous_out = mac['mass outflux [mol/d]'] + -mac['source/sink from gas phase [mol/d]']
    ax[1, 0].plot(time_yr, aqueous_in - aqueous_out, linestyle=':', label='Influx + production - outflux - gas partition')
    ax[1, 0].legend()
    ax[1, 0].set(ylabel='mol/d', title='Change in aqueous storage')
    ax[1, 0].annotate('(All three curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')

    # Calculate change in gas storage
    ax[1, 1].plot(time_yr, mgc['change in storage [mol/d]'], color='k', label='From mgc file')
    ax[1, 1].plot(time_yr[1:], np.diff(mgs['gtr(g)'].values)/np.diff(time_yr*365.25), color='0.5', linestyle='--', label='np.diff(mgs)')
    # Inputs = influx (from top) + gain from aqueous phase
    gas_in = mgc['mass influx [mol/d]'] + -mgc['source/sink - aqueous phase [mol/d]']
    # Outputs = outflux (from top)
    gas_out = mgc['mass outflux [mol/d]']
    ax[1, 1].plot(time_yr, gas_in - gas_out, linestyle=':', label='Influx + aq partition - outflux')
    ax[1, 1].legend()
    ax[1, 1].set(ylabel='mol/d', title='Change in gas storage')
    ax[1, 1].annotate('(All three curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')

    fig.suptitle(f'{site}: {sim_name}')

    return fig, ax

f, a = plot_gas_tracer_mass_balance('Yolo', 'erw', 'spinup', 35)

# Check that the CO$_2$ production rate is the intended rate

Note that this just checks mineral dissolution, and does not consider CO2 partitioning into the carbonate system

In [ ]:
import fsspec
from byte_util.reaction import co2_factors

# Load in and scale CO2 respiration
soil_resp_calc = 'GB94'
database_rate = 1e-13  # mol/m2/s (intrinsic rate constant specified in the database)
co2_scaling_factor = np.array(list(co2_factors.values()), dtype=np.float64)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'
s3_input_path = f'{s3_base_path}/input-data/processed-data'
k_co2_path = f'{s3_input_path}/soilco2production_profiles_{soil_resp_calc}.npy'

with fsspec.open(k_co2_path, 'rb') as f:
    k_co2 = np.load(f)
k_co2 *= co2_scaling_factor[:, np.newaxis]

for site in all_sites:
    print(site)

    # Calculate total rate in mol/s (across entire profile)
    # mol/s = mol/L aquifer/s * 1000 L/m3 dx (1 m) * dy (1 m) * dz (0.01 m)
    site_idx = all_sites.index(site)
    total_rate_mol_s = np.sum(k_co2[site_idx] * 1000 * 1 * 1 * 0.01)
    total_rate_mol_d = total_rate_mol_s * 60 * 60 * 24
    print(f'Target rate: {total_rate_mol_d:0.4f} mol/d')

    for forcing in ['spinup'] + all_forcing_types:
        for scenario in ['ctrl', 'erw']:

            if forcing == 'spinup':
                if scenario == 'erw':
                    continue
                sim_folder = sim_root / site / forcing
                sim_name = 'spinup'
            else:
                sim_folder = sim_root / site / f'{forcing}_{scenario}'
                sim_name = f'{forcing}_{scenario}'

            mmc, _ = read_min3p(sim_folder / f'{forcing}_1.mmc', ftype='transient', fmt='pandas')
            min_rate = -mmc['source/sink - aqueous phase [mol/d]'].min()
            max_rate = -mmc['source/sink - aqueous phase [mol/d]'].max()
            mean_rate = -mmc['source/sink - aqueous phase [mol/d]'].mean()

            mean_error = (mean_rate - total_rate_mol_d)/total_rate_mol_d
            print(f'\t{sim_name} min/max/mean: {min_rate:0.4f}/{max_rate:0.4f}/{mean_rate:0.4f} mol/d')
            print(f'\t{sim_name} mean error: {mean_error*100:.0f}%')

# Is the small difference in water content really enough to generate a 2x increase in pCO2?

In [ ]:
from min3p.output import read_min3p
import pandas as pd
from byte_util.met_forcing_plots import read_transient, cell_size_m, all_forcing_types, time_mean, temperature_K

# Plot surface diffusive flux
def calculate_theoretical_surface_co2_flux(site, forcing, scenario, gas_diffusivity=1.8e-5,
                                           boundary_conc=4.22e-4):
    gbp, gbp_cols, cells = read_transient(site, forcing, scenario, 'gbp')
    gbg, gbg_cols, _ = read_transient(site, forcing, scenario, 'gbg')
    # Remove 0th time step for gbg
    gbg = gbg[:, :, 1:]
    # Get indices of top and second cells
    top, second = np.argsort(cells)[-2:][::-1]

    theta_a = gbp[top, gbp_cols.index('theta_a')]
    theta_g = gbp[top, gbp_cols.index('theta_g')]
    porosity = theta_a + theta_g

    # molar_concentration (mol/m3) = conc_atm (atm) * 101325 Pa/atm / gas_constant (Pa m3/mol/K) / temperature (K)
    molar_conc = gbg[top, gbg_cols.index('co2(g)')] * 101325 / (8.314462618 * temperature_K)
    molar_conc_boundary = boundary_conc * 101325 / (8.314462618 * temperature_K)

    # Calculate tortuosity using Millington (1959)
    tau = (theta_g / porosity)**(7 / 3) * porosity**(1 / 3)

    # Diffusivity (m2/s) = Dg (m2/s) * average gas-filled volume (m3/m3) * tortuosity (-)
    effective_diffusivity = gas_diffusivity * theta_g * tau

    # Calculate gradient: molar_gradient ((mol/m3)/m) = dC (mol/m3) / dz (m)
    molar_gradient = (molar_conc - molar_conc_boundary) / (cell_size_m/2)

    return pd.DataFrame({
        'time_d': gbg[top, gbg_cols.index('time')],
        'theta_a': theta_a,
        'theta_g': theta_g,
        'porosity': porosity,
        'mean_conc': time_mean(molar_conc, gbg[top, gbg_cols.index('time')]),
        'effective_diffusivity_m2_s': effective_diffusivity,
        'efflux_mol_m2_d': 86400 * effective_diffusivity * molar_gradient,
    })


def plot_surface_co2_efflux(site, scenario='ctrl', figsize=(10, 6), forcing_types=None,
                            forcing_colors=None):
    if forcing_types is None:
        forcing_types = all_forcing_types[::-1]
    if forcing_colors is None:
        forcing_colors = dict(zip(forcing_types, plt.colormaps['Set2'](np.linspace(0.05, 0.95, 4))))

    fig, ax = plt.subplots(2, 3, figsize=figsize, sharex=True, tight_layout=True)

    # Calculate concentration gradient using the same concentrations to see if
    # water content alone is enough to account for difference in CO2 diffusion
    longterm_co2 = calculate_theoretical_surface_co2_flux(site, 'longterm', scenario)
    molar_conc_boundary = 4.22e-4 * 101325 / (8.314462618 * temperature_K)
    molar_conc_top_cell = longterm_co2['mean_conc'].values[0]
    molar_gradient = (molar_conc_top_cell - molar_conc_boundary) / (cell_size_m/2)

    for forcing in forcing_types:
        co2_efflux = calculate_theoretical_surface_co2_flux(site, forcing, scenario)
        co2_years = co2_efflux['time_d'] / 365
        efflux_mol_d = 86400 * co2_efflux['effective_diffusivity_m2_s'] * molar_gradient
        sim_folder = sim_root / site / f'{forcing}_{scenario}'

        mgc, _ = read_min3p(f'{forcing}_1.mgc', folder=sim_folder, fmt='pandas', ftype='transient')
        mgc_years = mgc['time [days]'] / 365
        ax[0, 0].plot(co2_years, co2_efflux['theta_a'], color=forcing_colors[forcing], label=forcing)
        ax[0, 1].plot(mgc_years, mgc['mass outflux [mol/d]'], color=forcing_colors[forcing])
        # ax[0, 2].plot(years, co2_efflux['efflux_mol_m2_d'], color=forcing_colors[forcing])
        ax[0, 2].plot(co2_years, efflux_mol_d, color=forcing_colors[forcing])
        mean_theta = time_mean(co2_efflux['theta_a'].values, co2_efflux['time_d'].values)
        mean_efflux = time_mean(mgc['mass outflux [mol/d]'].values, mgc['time [days]'].values)
        # mean_theoretical = time_mean(co2_efflux['efflux_mol_m2_d'].values, mgc['time [days]'].values)
        mean_theoretical = time_mean(efflux_mol_d.values, co2_efflux['time_d'].values)
        ax[1, 0].axhline(mean_theta, color=forcing_colors[forcing], ls='--', label=forcing)
        ax[1, 1].axhline(mean_efflux, color=forcing_colors[forcing], ls='--', label=forcing)
        ax[1, 2].axhline(mean_theoretical, color=forcing_colors[forcing], ls='--')

    for i in range(3):
        ax[1, i].set(xlabel='Time (y)')
    for j in range(2):
        ax[j, 0].set(ylabel='Top cell water content (m$^{3}$ m$^{-3}$)')
        ax[j, 1].set(ylabel='Actual CO$_2$ efflux (mol m$^{-2}$ d$^{-1}$)')
        ax[j, 2].set(ylabel='Theoretical CO$_2$ efflux (mol m$^{-2}$ d$^{-1}$)')
        ax[j, 0].legend()

    ax[1, 0].set(ylim=ax[0, 0].get_ylim())
    ax[1, 1].set(ylim=ax[0, 1].get_ylim())
    ax[1, 2].set(ylim=ax[0, 1].get_ylim())
    fig.suptitle(f'{site}: Diffusive CO$_2$ gas flux ({scenario.upper()})')
    return fig, ax


site = 'HoustonBlack'
scenario = 'ctrl'

fig, ax = plot_surface_co2_efflux(site, scenario='ctrl')

In [ ]:
# Plot mass balance for co2
def plot_co2_mass_balance(site, scenario, forcing, skip_records=0):
    """Plot mass balance for the gas tracer.

    skip_records : int
        Number of initial records to skip (to avoid very large values at the start of spin up)"""

    if forcing == 'spinup' or scenario == 'spinup':
        sim_folder = sim_root / site / 'spinup'
        sim_name = 'spinup'
    else:
        sim_folder = sim_root / site / f'{forcing}_{scenario}'
        sim_name = forcing

    # mas : total mol of aqueous species
    mas, _ = read_min3p(sim_folder / f'{sim_name}_o.mas', ftype='transient', fmt='pandas')
    # mgs : total mol of gaseous species
    mgs, _ = read_min3p(sim_folder / f'{sim_name}_o.mgs', ftype='transient', fmt='pandas')
    # mgc : gas mass balance
    mgc, _ = read_min3p(sim_folder / f'{sim_name}_1.mgc', ftype='transient', fmt='pandas')
    mac, _ = read_min3p(sim_folder / f'{sim_name}_2.mac', ftype='transient', fmt='pandas')
    mmc_co2_resp, _ = read_min3p(sim_folder / f'{sim_name}_1.mmc', ftype='transient', fmt='pandas')
    mmc_calcite, _ = read_min3p(sim_folder / f'{sim_name}_5.mmc', ftype='transient', fmt='pandas')
    mas.drop(0, inplace=True)
    mgs.drop(0, inplace=True)
    mas.index = np.arange(len(mas))
    mgs.index = np.arange(len(mgs))

    mask = mas['time'].duplicated()
    for df in [mas, mgs, mgc, mac, mmc_co2_resp, mmc_calcite]:
        # Remove duplicate timesteps (happens when dt is small relative to output precision
        df.drop(df.index[mask], inplace=True)
        df.index = np.arange(len(df))

        # If requested, skip initial records
        if skip_records > 0:
            df.drop(range(0, skip_records), inplace=True)

    time_yr = mas['time'].values/365.25

    fig, ax = plt.subplots(2, 3, figsize=(12, 6), sharex='all')

    # Plot total storage
    ax[0, 0].plot(time_yr, mas['co3-2'], label='Aqueous')
    ax[0, 0].plot(time_yr, mgs['co2(g)'], label='Gaseous')
    ax[0, 0].plot(time_yr, mas['co3-2'] + mgs['co2(g)'], label='Total', color='k', ls='--')
    ax[0, 0].set(ylabel='mol', title='Total in domain')
    ax[0, 0].legend()

    # Dissolution of calcite and co2
    ax[0, 1].plot(time_yr, -mmc_co2_resp['source/sink - aqueous phase [mol/d]'], label='Mineral dissolution (CO2)', color='k')
    ax[0, 1].plot([5]*2, [-mmc_co2_resp['source/sink - aqueous phase [mol/d]'].mean()]*2, label='Mineral dissolution (calcite)', color='0.5', ls='--')
    ax1a = ax[0, 1].twinx()
    ax1a.plot(time_yr, -mmc_calcite['source/sink - aqueous phase [mol/d]'], label='Mineral dissolution (calcite)', color='0.5', ls='--')
    ax[0, 1].set(ylabel='mol/d', title='Mineral dissolution')
    ax[0, 1].legend()

    # Total aqueous production
    # Total minearl = co2_resp + calcite
    total_mineral = -mmc_co2_resp['source/sink - aqueous phase [mol/d]'] + -mmc_calcite['source/sink - aqueous phase [mol/d]']
    ax[0, 2].plot(time_yr, total_mineral, label='Mineral dissolution (mmc)', color='0.5', ls='--')
    ax[0, 2].plot(time_yr, mac['source/sink from mineral phase [mol/d]'], label='Mineral production (mac)', color='0.5', ls='--')
    ax[0, 2].set(ylabel='mol/d', title='Total mineral production')
    ax[0, 2].annotate('(Both curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')
    ax[0, 1].legend()

    # Calculate change in aqueous storage
    ax[1, 0].plot(time_yr, mac['change in storage [mol/d]'], color='k', label='From mac file')
    ax[1, 0].plot(time_yr[1:], np.diff(mas['co3-2'].values)/np.diff(time_yr*365.25), color='0.5', linestyle='--', label='np.diff(mas)')
    # Inputs = inflow from top + dissolution rate
    aqueous_in = mac['mass influx [mol/d]'] + mac['source/sink from mineral phase [mol/d]']
    # Outputs = outflow from bottom + loss to gas phase
    aqueous_out = mac['mass outflux [mol/d]'] + -mac['source/sink from gas phase [mol/d]']
    ax[1, 0].plot(time_yr, aqueous_in - aqueous_out, linestyle=':', label='Influx + production - outflux - gas partition')
    ax[1, 0].legend()
    ax[1, 0].set(ylabel='mol/d', title='Change in aqueous storage', xlabel='Time (y)')
    ax[1, 0].annotate('(All three curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')

    # Calculate change in gas storage
    ax[1, 1].plot(time_yr, mgc['change in storage [mol/d]'], color='k', label='From mgc file')
    ax[1, 1].plot(time_yr[1:], np.diff(mgs['co2(g)'].values)/np.diff(time_yr*365.25), color='0.5', linestyle='--', label='np.diff(mgs)')
    # Inputs = influx (from top) + gain from aqueous phase
    gas_in = mgc['mass influx [mol/d]'] + -mgc['source/sink - aqueous phase [mol/d]']
    # Outputs = outflux (from top)
    gas_out = mgc['mass outflux [mol/d]']
    ax[1, 1].plot(time_yr, gas_in - gas_out, linestyle=':', label='Influx + aq partition - outflux')
    ax[1, 1].legend()
    ax[1, 1].set(ylabel='mol/d', title='Change in gas storage', xlabel='Time (y)')
    ax[1, 1].annotate('(All three curves\nshould overlap)', xy=(0.98, 0.5), xycoords='axes fraction', ha='right', va='center', fontsize=8, fontstyle='italic')

    fig.suptitle(f'{site}: {sim_name} ({scenario})')

    return fig, ax

f, a = plot_co2_mass_balance('Yolo', 'ctrl', 'hourly', 35)

In [ ]:
from scipy.integrate import cumulative_trapezoid

# Plot mass balance for co2

from min3p.output import read_min3p

def interp_common_time(t1, y1, t2, y2, dt=1):
    tmin = max(t1.min(), t2.min())
    tmax = min(t1.max(), t2.max())

    t = np.arange(tmin, tmax+dt, dt)

    return t, np.interp(t, t1, y1), np.interp(t, t2, y2)

def plot_cumulative_co2_flux(site, scenario, forcing1='longterm', forcing2='hourly', sharey='row'):
    """Plot mass balance for the gas tracer.

    skip_records : int
        Number of initial records to skip (to avoid very large values at the start of spin up)"""

    f1color = 'steelblue'
    f2color = 'firebrick'

    f1_folder = sim_root / site / f'{forcing1}_{scenario}'
    f2_folder = sim_root / site / f'{forcing2}_{scenario}'

    # mgs : total mol of gaseous species
    f1_mgs, _ = read_min3p(f1_folder / f'{forcing1}_o.mgs', ftype='transient', fmt='pandas')
    f2_mgs, _ = read_min3p(f2_folder / f'{forcing2}_o.mgs', ftype='transient', fmt='pandas')

    # mas : total mol of aqueous species
    f1_mas, _ = read_min3p(f1_folder / f'{forcing1}_o.mas', ftype='transient', fmt='pandas')
    f2_mas, _ = read_min3p(f2_folder / f'{forcing2}_o.mas', ftype='transient', fmt='pandas')

    # mgc : gas mass balance
    f1_mgc, _ = read_min3p(f1_folder / f'{forcing1}_1.mgc', ftype='transient', fmt='pandas')
    f2_mgc, _ = read_min3p(f2_folder / f'{forcing2}_1.mgc', ftype='transient', fmt='pandas')

    # mac : aqueous mass balance
    f1_mac, _ = read_min3p(f1_folder / f'{forcing1}_2.mac', ftype='transient', fmt='pandas')
    f2_mac, _ = read_min3p(f2_folder / f'{forcing2}_2.mac', ftype='transient', fmt='pandas')

    for mgs in [f1_mgs, f2_mgs, f1_mas, f2_mas]:
        mgs.drop(0, inplace=True)
        mgs.index = np.arange(len(mgs))

    # Interpolate to consistent time
    time_d = np.arange(0, 3650 + 1, 1)
    time_yr = time_d / 365
    f1_co2 = np.interp(time_d, f1_mgs['time'].values, f1_mgs['co2(g)'].values)
    f2_co2 = np.interp(time_d, f2_mgs['time'].values, f2_mgs['co2(g)'].values)
    f1_aq = np.interp(time_d, f1_mas['time'].values, f1_mas['co3-2'].values)
    f2_aq = np.interp(time_d, f2_mas['time'].values, f2_mas['co3-2'].values)
    f1_aq_efflux = cumulative_trapezoid(f1_mac['mass outflux [mol/d]'], f1_mac['time [days]'], initial=0)
    f2_aq_efflux = cumulative_trapezoid(f2_mac['mass outflux [mol/d]'], f2_mac['time [days]'], initial=0)
    f1_aq_interp = np.interp(time_d, f1_mac['time [days]'].values, f1_aq_efflux)
    f2_aq_interp = np.interp(time_d, f2_mac['time [days]'].values, f2_aq_efflux)

    fig, ax = plt.subplots(2, 3, figsize=(12, 6), tight_layout=True, sharex='all', sharey=sharey)

    # Plot total storage
    ax[0, 0].plot(time_yr, f1_co2, color=f1color, label=forcing1)
    ax[0, 0].plot(time_yr, f2_co2, color=f2color, label=forcing2)
    ax[0, 1].plot(time_yr, f1_aq, color=f1color)
    ax[0, 1].plot(time_yr, f2_aq, color=f2color)
    ax[0, 2].plot(time_yr, (f1_co2 + f1_aq) - (f2_co2 + f2_aq), color='k')
    ax[0, 0].set(ylabel='Total gaseous CO$_2$ (mol)', title='Gas')
    ax[0, 1].set(ylabel='Total aqueous CO$_2$ (mol)', title='Aqueous')
    ax[0, 2].set(ylabel='Diff. in total CO$_2$ (a + g; mol)', title=f'{forcing1} - {forcing2}')
    ax[0, 0].legend()

    # Plot cumulative efflux
    f1_cumulative_efflux = cumulative_trapezoid(f1_mgc['mass outflux [mol/d]'], f1_mgc['time [days]'], initial=0)
    ax[1, 0].plot(f1_mgc['time [days]']/365, f1_cumulative_efflux, color=f1color)
    f2_cumulative_efflux = cumulative_trapezoid(f2_mgc['mass outflux [mol/d]'], f2_mgc['time [days]'], initial=0)
    ax[1, 0].plot(f2_mgc['time [days]']/365, f2_cumulative_efflux, color=f2color)
    ax[1, 1].plot(f1_mac['time [days]']/365, f1_aq_efflux, color=f1color)
    ax[1, 1].plot(f2_mac['time [days]']/365, f2_aq_efflux, color=f2color)
    f1_efflux_interp = np.interp(time_d, f1_mgc['time [days]'].values, f1_cumulative_efflux)
    f2_efflux_interp = np.interp(time_d, f2_mgc['time [days]'].values, f2_cumulative_efflux)
    ax[1, 2].plot(time_yr, f1_efflux_interp - f2_efflux_interp, color='k')
    ax[1, 0].set(ylabel='Cum. gas efflux (mol)')
    ax[1, 1].set(ylabel='Cum. aq. efflux (mol)')
    ax[1, 2].set(ylabel='Diff. in gas efflux (mol)')
    for i in range(3):
        ax[1, i].set(xlabel='Time (y)')

    fig.suptitle(f'{site}: {forcing1} vs {forcing2} CO$_2$ comparison ({scenario})')

    return fig, ax

f, a = plot_cumulative_co2_flux('Yolo', 'ctrl', sharey=False)

# Check intended vs actual met forcing fluxes

In [ ]:
# Calculate initial surface flux for each site and simulation type
from byte_util.util import start_date
from byte_util.met_transport import create_bcvs_soi
from byte_util.met_forcing_plots import cell_size_m, time_mean

def calc_intended_fluxes(site, forcing_type, cell_size_m=cell_size_m):
    forcing_file = f'{s3_input_path}/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Limit to 10-year simulation period
    end_date = pd.to_datetime(start_date) + pd.Timedelta(days=3651)
    met_forcing = met_forcing.loc[start_date:end_date, :]

    # Convert units and time average using create_bcvs_soi
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24
    bcvs, soi = create_bcvs_soi(met_forcing, freq=forcing_type, start_date=start_date)

    # Convert surface flux back to m/d; convert transpiration from 1/d to m/d
    bcvs['surface_flux_m.d'] = bcvs['surface_flux_m.s'] * 60 * 60 * 24
    soi['transpiration_m.d'] = soi['transpiration_factor'] * cell_size_m

    return bcvs, soi


site = 'Yolo'
forcing_type = 'daily'
scenario = 'ctrl'
sim_folder = sim_root / site / f'{forcing_type}_{scenario}'
mvc, _ = read_min3p(f'{forcing_type}_o.mvc', folder=sim_folder, fmt='pandas', ftype='transient')
bcvs, soi = calc_intended_fluxes(site, forcing_type)

fig, ax = plt.subplots(2, figsize=(8, 5), sharex=True, tight_layout=True)

ax[0].plot(bcvs['time'], bcvs['surface_flux_m.d'], color='steelblue', label='Intended value')
ax[1].plot(soi['time'], soi['transpiration_m.d'], color='forestgreen', label='Intended value')

ax[0].plot(mvc['time'], mvc['inflow'], color='0.3', ls='--', label='Actual value')
ax[1].plot(mvc['time'], mvc['root water uptake'], color='0.3', ls='--', label='Actual value')

ax[0].set(ylabel='Surface flux (m/d)', title=f'{site}: {forcing_type}_{scenario}')
ax[1].set(ylabel='Transpiration (m/d)', xlabel='Time (d)')

# Print mean stats
mean_comparions = [(time_mean(bcvs['surface_flux_m.d'].values, bcvs['time'].values),
                    time_mean(mvc['inflow'].values, mvc['time'].values)),
                   (time_mean(soi['transpiration_m.d'].values, soi['time'].values),
                    time_mean(mvc['root water uptake'].values, mvc['time'].values))]
for i, (intended, actual) in enumerate(mean_comparions):
    pct_diff = (actual - intended)/intended*100
    ax[i].annotate(f'Actual-Intended: {pct_diff:.0f}% m/d', (0.01, 0.92), xycoords='axes fraction', ha='left')
    ax[i].legend()

ax[1].set(xlim=[1000, 1100])

### Compare mean surface inflow values across each forcing type

In [ ]:
# Comparison of mean bcvs values (can't use mvc because that combines inflow and outflow)
scenario = 'erw'
reference_forcing = 'longterm'

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True, tight_layout=True)
width = 0.18
x = np.arange(len(all_sites))
surface_flux = {}

forcing_colors = dict(zip(all_forcing_types, plt.colormaps['Set2'](np.linspace(0.05, 0.95, 4))))

for site in all_sites:
    for forcing in all_forcing_types:
        sim_folder = sim_root / site / f'{forcing}_{scenario}'
        if forcing == 'longterm':
            infile = InputFile.load(sim_folder / f'{forcing}.dat')
            sf_m_s = infile.boundary_conditions_vsflow.zones[0].boundary_value
            surface_flux[site, forcing] = sf_m_s * 60 * 60 * 24
        else:
            bcvs, _ = calc_intended_fluxes(site, forcing)
            surface_flux[site, forcing] = time_mean(bcvs['surface_flux_m.d'].values, bcvs['time'].values)

for j, forcing in enumerate(all_forcing_types):
    values = np.array([surface_flux[site, forcing] for site in all_sites])
    reference = np.array([surface_flux[site, reference_forcing] for site in all_sites])
    error = 100 * (values - reference) / reference

    pos = x + (j - (len(all_forcing_types) - 1) / 2) * width
    ax[0].bar(pos, values, width, color=forcing_colors[forcing], label=forcing, edgecolor='k')
    ax[1].bar(pos, error, width, color=forcing_colors[forcing], edgecolor='k')

ax[0].set(ylabel=f'Mean surface flux (m/d)', title=f'Inflow comparison for {scenario}')
ax[1].set(ylabel=f'Error relative to {reference_forcing} (%)', xticks=x, xticklabels=all_sites)
ax[1].axhline(0, color='k', lw=0.8)
ax[0].legend()


In [ ]:
# Same as above, but using bcvs file
scenario = 'erw'
reference_forcing = 'longterm'

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True, tight_layout=True)
width = 0.18
x = np.arange(len(all_sites))
surface_flux = {}

forcing_colors = dict(zip(all_forcing_types, plt.colormaps['Set2'](np.linspace(0.05, 0.95, 4))))

for site in all_sites:
    for forcing in all_forcing_types:
        sim_folder = sim_root / site / f'{forcing}_{scenario}'
        if forcing == 'longterm':
            infile = InputFile.load(sim_folder / f'{forcing}.dat')
            sf_m_s = infile.boundary_conditions_vsflow.zones[0].boundary_value
            surface_flux[site, forcing] = sf_m_s * 60 * 60 * 24
        else:
            bcvs = pd.read_csv(sim_folder/f'{forcing}.bcvs',
                               names=['time', 'surface_flux_m.s', 'null'], sep=r'\s+')
            bcvs['surface_flux_m.d'] = bcvs['surface_flux_m.s'] * 60 * 60 * 24
            surface_flux[site, forcing] = time_mean(bcvs['surface_flux_m.d'].values, bcvs['time'].values)

for j, forcing in enumerate(all_forcing_types):
    values = np.array([surface_flux[site, forcing] for site in all_sites])
    reference = np.array([surface_flux[site, reference_forcing] for site in all_sites])
    error = 100 * (values - reference) / reference

    pos = x + (j - (len(all_forcing_types) - 1) / 2) * width
    ax[0].bar(pos, values, width, color=forcing_colors[forcing], label=forcing, edgecolor='k')
    ax[1].bar(pos, error, width, color=forcing_colors[forcing], edgecolor='k')

ax[0].set(ylabel=f'Mean surface flux (m/d)', title=f'Inflow comparison for {scenario}')
ax[1].set(ylabel=f'Error relative to {reference_forcing} (%)', xticks=x, xticklabels=all_sites)
ax[1].axhline(0, color='k', lw=0.8)
ax[0].legend()


### Compare mean root water uptake values across each forcing type

Note that we specify the same potential transpiration for each simulation, but the root water uptake function will adjust transpiration dynamically based on water content. Thus, if hourly water content is on average lower than longterm water content, the root water uptake should be lower as well.

In [ ]:
# Comparison of mean bcvs values (can't use mvc because that combines inflow and outflow)
scenario = 'erw'
reference_forcing = 'longterm'
value_to_plot = 'root water uptake'

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True, tight_layout=True)
width = 0.18
x = np.arange(len(all_sites))
mvc_values = {}

forcing_colors = dict(zip(all_forcing_types, plt.colormaps['Set2'](np.linspace(0.05, 0.95, 4))))

for site in all_sites:
    for forcing in all_forcing_types:
        sim_folder = sim_root / site / f'{forcing}_{scenario}'
        mvc, _ = read_min3p(f'{forcing}_o.mvc', folder=sim_folder, fmt='pandas', ftype='transient')
        mvc_values[site, forcing] = time_mean(mvc[value_to_plot].values, mvc['time'].values)

for j, forcing in enumerate(all_forcing_types):
    values = np.array([mvc_values[site, forcing] for site in all_sites])
    reference = np.array([mvc_values[site, reference_forcing] for site in all_sites])
    error = 100 * (values - reference) / reference

    pos = x + (j - (len(all_forcing_types) - 1) / 2) * width
    ax[0].bar(pos, values, width, color=forcing_colors[forcing], label=forcing, edgecolor='k')
    ax[1].bar(pos, error, width, color=forcing_colors[forcing], edgecolor='k')

ax[0].set(ylabel=f'Mean {value_to_plot} (m/d)', title=f'{value_to_plot} comparison for {scenario}')
ax[1].set(ylabel=f'Error relative to {reference_forcing} (%)', xticks=x, xticklabels=all_sites)
ax[1].axhline(0, color='k', lw=0.8)
ax[0].legend()

# Check that we applied the right amount for forsterite

Ideally, this should be 10 t/ha

In [ ]:
from min3p.output import read_min3p
from byte_util.reaction import mineral_params
from byte_util.util import dz

dx, dy = 1.0, 1.0

for site in all_sites:
    for forcing_type in all_forcing_types:
        ctrl_min, min_cols, _ = read_min3p(f'{forcing}_0.gsv', folder=sim_root / site / f'{forcing}_ctrl')
        erw_min, min_cols, _ = read_min3p(f'{forcing}_0.gsv', folder=sim_root / site / f'{forcing}_erw')

        # mask out feedstock application depth
        fidx = min_cols.index('forst-ph')
        mask = erw_min[fidx] > ctrl_min[fidx]
        ctrl_min[fidx, mask] = 0.0

        cell_volumes = np.ones((erw_min.shape[-1],), dtype=float)
        cell_volumes *= dx*dy*dz
        applied_volumes = erw_min[fidx]*cell_volumes - ctrl_min[fidx]*cell_volumes
        total_volume_m3 = np.sum(applied_volumes)

        # Convert total volume (in m3) to tonnes
        forsterite_density_g_cm3 = mineral_params['forst-ph']['density']
        forsterite_density_kg_m3 = forsterite_density_g_cm3 * 1000
        total_mass_kg = total_volume_m3 * forsterite_density_kg_m3
        total_mass_tonnes = total_mass_kg / 1000

        # Calculate area in hectares
        area_m2 = dx * dy
        area_ha = area_m2 / 10000

        # Calculate tonnes per hectare
        tonnes_per_ha = total_mass_tonnes / area_ha
        print(f'{site} ({forcing}): Applied forsterite = {tonnes_per_ha:.2f} t/ha')